In [2]:
# =====================================================
# VGG16 on HAM10000 (7-class skin lesion classification)
# Run in a Kaggle notebook with GPU ON.
# Dataset to add: "Skin Cancer MNIST: HAM10000" (kmader)
# Internet must be ON once, to download the pretrained VGG16 weights.
# =====================================================

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

# ---------------- 1. Settings ----------------
SEED = 42
IMG_SIZE = 224          # VGG16 expects 224x224
BATCH_SIZE = 32
EPOCHS_HEAD = 3         # phase 1: train only the new classifier layer
EPOCHS_FINE = 10        # phase 2: also fine-tune the last conv blocks
LR_HEAD = 1e-3
LR_FINE = 1e-4

# Kaggle mounts datasets at different paths depending on the notebook version,
# so instead of hard-coding the path we search for the metadata file.
csv_matches = glob.glob("/kaggle/input/**/HAM10000_metadata*.csv", recursive=True)
if not csv_matches:
    print("Could not find HAM10000_metadata.csv. Here is what is in /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, "->", files[:5])
        if root.count(os.sep) > 5:
            break
    raise FileNotFoundError(
        "HAM10000 not found. In the notebook, click 'Add Input' (right sidebar) "
        "and add the 'Skin Cancer MNIST: HAM10000' dataset, then re-run.")
METADATA_CSV = csv_matches[0]
DATA_DIR = os.path.dirname(METADATA_CSV)
print("Found metadata at:", METADATA_CSV)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- 2. Load metadata ----------------
df = pd.read_csv(METADATA_CSV)

# Map every image_id to its file path (works even if the folders are nested)
all_paths = glob.glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True)
id_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in all_paths}
df["path"] = df["image_id"].map(id_to_path)
df = df.dropna(subset=["path"]).reset_index(drop=True)

# Class names -> numbers
classes = sorted(df["dx"].unique())          # akiec, bcc, bkl, df, mel, nv, vasc
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label"] = df["dx"].map(class_to_idx)
print("Classes:", class_to_idx)
print(df["dx"].value_counts())

# ---------------- 3. Train / val / test split ----------------
# IMPORTANT: split by lesion_id. The same lesion can have several photos,
# and if they land in both train and test, the score becomes unrealistically high.
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df, groups=df["lesion_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["lesion_id"]))
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ---------------- 4. Dataset + transforms ----------------
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet values

train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


class HAMDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img), int(row["label"])


train_loader = DataLoader(HAMDataset(train_df, train_tfm), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(HAMDataset(val_df, eval_tfm), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(HAMDataset(test_df, eval_tfm), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

# ---------------- 5. Model: pretrained VGG16 ----------------
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
model.classifier[6] = nn.Linear(4096, len(classes))   # replace last layer: 1000 -> 7
model = model.to(device)

# ---------------- 6. Loss (with class weights for imbalance) ----------------
# 'nv' (moles) is ~67% of the data, so rare classes get a bigger weight.
counts = train_df["label"].value_counts().sort_index().values
weights = counts.sum() / (len(counts) * counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


# ---------------- 7. Train / evaluate helpers ----------------
def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


best_val_acc = 0.0


def fit(epochs, optimizer, tag):
    global best_val_acc
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(train_loader, optimizer)
        va_loss, va_acc = run_epoch(val_loader)
        print(f"[{tag}] Epoch {ep}/{epochs} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f}")
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), "best_vgg16_ham10000.pth")
            print("   -> saved best model")


# ---------------- 8. Phase 1: train only the new head ----------------
for p in model.features.parameters():
    p.requires_grad = False                      # freeze convolution layers
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
fit(EPOCHS_HEAD, opt, "head")

# ---------------- 9. Phase 2: fine-tune last conv blocks ----------------
# VGG16 features[0:17] = first 3 blocks (generic edges/textures) -> keep frozen
# VGG16 features[17:] = last 2 blocks (more task-specific) -> unfreeze
for p in model.features[17:].parameters():
    p.requires_grad = True
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINE)
fit(EPOCHS_FINE, opt, "fine")

# ---------------- 10. Final test evaluation ----------------
model.load_state_dict(torch.load("best_vgg16_ham10000.pth"))
model.eval()
preds, targets = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            out = model(x)
        preds.extend(out.argmax(1).cpu().numpy())
        targets.extend(y.numpy())

print("\nTest results:")
print(classification_report(targets, preds, target_names=classes, digits=4))
print("Confusion matrix:")
print(confusion_matrix(targets, preds))

Found metadata at: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv
Device: cuda
Classes: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Train: 7002 | Val: 1519 | Test: 1494
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 218MB/s]  


[head] Epoch 1/3 | train loss 3.0830 acc 0.3330 | val loss 1.2523 acc 0.6234
   -> saved best model
[head] Epoch 2/3 | train loss 3.0298 acc 0.2724 | val loss 1.4289 acc 0.4839
[head] Epoch 3/3 | train loss 2.5421 acc 0.2064 | val loss 1.4615 acc 0.5839
[fine] Epoch 1/10 | train loss 1.9157 acc 0.3323 | val loss 1.3838 acc 0.6175
[fine] Epoch 2/10 | train loss 1.6834 acc 0.4829 | val loss 1.4648 acc 0.5313
[fine] Epoch 3/10 | train loss 1.3380 acc 0.5650 | val loss 1.0382 acc 0.5754
[fine] Epoch 4/10 | train loss 1.2237 acc 0.5508 | val loss 1.0537 acc 0.6011
[fine] Epoch 5/10 | train loss 1.1510 acc 0.5777 | val loss 1.1650 acc 0.4997
[fine] Epoch 6/10 | train loss 1.0369 acc 0.5887 | val loss 0.8862 acc 0.6359
   -> saved best model
[fine] Epoch 7/10 | train loss 0.9956 acc 0.6067 | val loss 0.8871 acc 0.6215
[fine] Epoch 8/10 | train loss 0.8683 acc 0.6440 | val loss 1.1027 acc 0.5622
[fine] Epoch 9/10 | train loss 0.8508 acc 0.6391 | val loss 1.0601 acc 0.5853
[fine] Epoch 10/10 | 